# Almgren-Chriss Market Impact Model Demonstration

This notebook demonstrates the Almgren-Chriss market impact model, which separates price impact into permanent and temporary components.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the parent directory to the path so we can import the modules
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))
from trade_simulator.src.orderbook import OrderBook
from trade_simulator.models.impact import (
    permanent_impact, 
    temporary_impact, 
    calculate_market_volume,
    calculate_total_impact
)

# Set random seed for reproducibility
np.random.seed(42)

## Generate Synthetic Orderbook Data

We'll create a function to generate synthetic orderbook data with varying levels of liquidity.

In [ ]:
def generate_synthetic_orderbook(base_price=50000.0, depth=10, spread=10.0, liquidity_factor=1.0):
    """
    Generate a synthetic orderbook with specified parameters.
    
    Args:
        base_price (float): The base price for the asset
        depth (int): Number of levels to generate on each side
        spread (float): The spread between best bid and best ask
        liquidity_factor (float): Factor to scale the liquidity (higher = more liquidity)
        
    Returns:
        OrderBook: A populated OrderBook instance
    """
    book = OrderBook("BTC-USDT-SWAP")
    
    # Calculate best bid and ask prices
    best_ask = base_price + spread / 2
    best_bid = base_price - spread / 2
    
    # Generate ask levels (ascending prices)
    asks = []
    for i in range(depth):
        price = best_ask + i * 10.0  # Price increases by $10 per level
        size = (1.0 + i * 0.5) * liquidity_factor  # Size increases with price
        asks.append([str(price), str(size)])
    
    # Generate bid levels (descending prices)
    bids = []
    for i in range(depth):
        price = best_bid - i * 10.0  # Price decreases by $10 per level
        size = (1.0 + i * 0.5) * liquidity_factor  # Size increases with depth
        bids.append([str(price), str(size)])
    
    # Create a tick with the generated data
    tick = {
        "timestamp": "2023-01-01T00:00:00Z",
        "exchange": "OKX",
        "symbol": "BTC-USDT-SWAP",
        "asks": asks,
        "bids": bids
    }
    
    book.update_from_tick(tick)
    return book

## Generate Price History for Volatility Calculation

We need to generate a price history to calculate volatility, which is a key input to the Almgren-Chriss model.

In [ ]:
def generate_price_history(book, n_ticks=100, volatility=0.001):
    """
    Generate a price history for volatility calculation.
    
    Args:
        book (OrderBook): The orderbook to update
        n_ticks (int): Number of ticks to generate
        volatility (float): Volatility parameter for price generation
    """
    base_price = book.mid_price()
    
    for i in range(n_ticks):
        # Generate a random price change
        price_change = np.random.normal(0, volatility * base_price)
        new_price = base_price + price_change
        
        # Update the orderbook with a new tick
        tick = {
            "timestamp": f"2023-01-01T00:{i//60:02d}:{i%60:02d}Z",
            "exchange": "OKX",
            "symbol": "BTC-USDT-SWAP",
            "asks": [[str(new_price + 5.0), "1.0"]],
            "bids": [[str(new_price - 5.0), "1.0"]]
        }
        
        book.update_from_tick(tick)
    
    return book

## Create Orderbook and Calculate Impact

Now we'll create an orderbook, generate a price history, and calculate the market impact for different order sizes.

In [ ]:
# Create a synthetic orderbook
book = generate_synthetic_orderbook(base_price=50000.0, depth=10, spread=10.0, liquidity_factor=1.0)

# Generate price history for volatility calculation
book = generate_price_history(book, n_ticks=100, volatility=0.001)

# Print orderbook state
print(book)

# Calculate market volume
market_volume = calculate_market_volume(book)
print(f"Market volume: {market_volume:.2f} BTC")

# Get volatility
volatility = book.rolling_volatility()
print(f"Volatility: {volatility:.2f} USD")

## Calculate Impact for Different Order Sizes

Let's calculate the permanent and temporary impact for different order sizes.

In [ ]:
# Define order sizes to test
order_sizes = np.linspace(0.1, 10.0, 20)  # From 0.1 BTC to 10 BTC

# Calculate impacts
perm_impacts = []
temp_impacts = []
total_impacts = []

for size in order_sizes:
    try:
        perm, temp, total = calculate_total_impact(book, size)
        perm_impacts.append(perm)
        temp_impacts.append(temp)
        total_impacts.append(total)
    except ValueError as e:
        print(f"Error for order size {size}: {e}")
        break

# Convert to numpy arrays
perm_impacts = np.array(perm_impacts)
temp_impacts = np.array(temp_impacts)
total_impacts = np.array(total_impacts)

## Visualize the Results

Let's plot the relationship between order size and market impact.

In [ ]:
plt.figure(figsize=(12, 6))

# Plot impacts
plt.plot(order_sizes, perm_impacts, 'b-', linewidth=2, label='Permanent Impact')
plt.plot(order_sizes, temp_impacts, 'r-', linewidth=2, label='Temporary Impact')
plt.plot(order_sizes, total_impacts, 'g-', linewidth=2, label='Total Impact')

plt.xlabel('Order Size (BTC)')
plt.ylabel('Impact (USD)')
plt.title('Almgren-Chriss Market Impact Model')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## Impact as Percentage of Order Value

Let's also calculate the impact as a percentage of the order value.

In [ ]:
# Calculate order values
mid_price = book.mid_price()
order_values = order_sizes * mid_price

# Calculate impact percentages
perm_impact_pcts = (perm_impacts / order_values) * 100
temp_impact_pcts = (temp_impacts / order_values) * 100
total_impact_pcts = (total_impacts / order_values) * 100

# Plot impact percentages
plt.figure(figsize=(12, 6))

plt.plot(order_sizes, perm_impact_pcts, 'b-', linewidth=2, label='Permanent Impact')
plt.plot(order_sizes, temp_impact_pcts, 'r-', linewidth=2, label='Temporary Impact')
plt.plot(order_sizes, total_impact_pcts, 'g-', linewidth=2, label='Total Impact')

plt.xlabel('Order Size (BTC)')
plt.ylabel('Impact (% of Order Value)')
plt.title('Almgren-Chriss Market Impact as Percentage of Order Value')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## Impact with Different Liquidity Levels

Let's compare the impact for different liquidity levels.

In [ ]:
# Define liquidity factors
liquidity_factors = [0.5, 1.0, 2.0]  # Low, medium, high liquidity
order_sizes = np.linspace(0.1, 5.0, 20)  # From 0.1 BTC to 5 BTC

plt.figure(figsize=(12, 6))

for factor in liquidity_factors:
    # Create a new orderbook with this liquidity
    book = generate_synthetic_orderbook(liquidity_factor=factor)
    book = generate_price_history(book, n_ticks=100, volatility=0.001)
    
    # Calculate impacts
    total_impacts = []
    for size in order_sizes:
        try:
            _, _, total = calculate_total_impact(book, size)
            total_impacts.append(total)
        except ValueError:
            break
    
    # Plot impacts
    plt.plot(order_sizes[:len(total_impacts)], total_impacts, 
             linewidth=2, label=f'Liquidity Factor: {factor}')

plt.xlabel('Order Size (BTC)')
plt.ylabel('Total Impact (USD)')
plt.title('Market Impact with Different Liquidity Levels')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## Conclusion

The Almgren-Chriss model provides a framework for understanding and quantifying market impact. Key observations:

1. Permanent impact grows with the square root of order size, while temporary impact grows linearly.
2. Total impact is the sum of permanent and temporary components.
3. Impact as a percentage of order value decreases with order size for permanent impact but remains constant for temporary impact.
4. Market liquidity significantly affects the magnitude of impact.